# Final Preprocessing – SMILES to Molecular Graphs

This notebook transforms chemical text into graph structures and prepares three learning stages so a GNN can understand molecules first, interactions second, and finally predict the toxicity of drug combinations.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install rdkit
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

  Using cached rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.8 kB)
Using cached rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl (36.7 MB)
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu118.html
  Using cached torch_scatter-2.1.2.tar.gz (108 kB)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.2 MB/s eta 0:00:00
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677287 sha256=e608b69813a5082fb9ee543b34b84c2ffb0de514ef18de67148bfbdf2383739b
  Stored in directory: /root/.cache/pip/wheels/84/20/50/448007

## Define Atom and Bond Feature Functions

In [3]:
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

def atom_features(atom):
    """
    Extract a feature vector for an atom.
    Features (11 dimensions):
      - Atom symbol one-hot (C, N, O, S, F, P, Cl, Br, I, B) -> 10 dimensions
      - Degree (one-hot up to 6) -> 7 dimensions
      - Formal charge (continuous, will be scaled later)
      - Hybridization (sp, sp2, sp3) -> 3 dimensions
      - Aromaticity (binary)
      - Atomic mass (continuous)
      - Number of hydrogens (continuous)
    """

    # Atom Symbol (What element is it?)
    # One-hot for atom symbol (common elements)
    symbol = atom.GetSymbol()
    symbols = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I', 'B']
    symbol_vec = [1.0 if symbol == s else 0.0 for s in symbols]

    # Degree (How many atoms are connected to it?)
    # Degree (up to 6)
    degree = atom.GetDegree()
    degree_vec = [0.0]*7
    degree_vec[min(degree, 6)] = 1.0

    # Formal Charge (Electrical charge)
    formal_charge = atom.GetFormalCharge()

    # Hybridization (Shape of electron orbitals)
    '''This changes molecule stability and reactions.'''
    hyb = atom.GetHybridization()
    hyb_vec = [
        1.0 if hyb == Chem.rdchem.HybridizationType.SP else 0.0,
        1.0 if hyb == Chem.rdchem.HybridizationType.SP2 else 0.0,
        1.0 if hyb == Chem.rdchem.HybridizationType.SP3 else 0.0
    ]

    # 5) Aromaticity (Special stable ring?)
    '''Aromatic atoms belong to special stable rings (like benzene).
    These rings strongly affect drug behavior and toxicity.'''
    is_aromatic = 1.0 if atom.GetIsAromatic() else 0.0

    # Atomic mass
    '''Heavier atoms often increase toxicity and bioactivity'''
    mass = atom.GetMass()

    # Number of hydrogens (implicit + explicit)
    '''Number of hydrogens affects solubility and reactivity'''
    num_h = atom.GetTotalNumHs()

    features = symbol_vec + degree_vec + [formal_charge] + hyb_vec + [is_aromatic, mass, num_h]
    return np.array(features, dtype=np.float32)

def bond_features(bond):
    """
    Extract features for a bond.
    Features (5 dimensions):
      - Bond type one-hot (single, double, triple, aromatic) -> 4 dimensions
      - Conjugation (binary)
      - In ring (binary)
    """
    bond_type = bond.GetBondType() # Bond Type (strength of connection)
    '''Bond type changes: molecule stability, reactivity, toxicity'''
    bond_type_vec = [
        1.0 if bond_type == Chem.rdchem.BondType.SINGLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.DOUBLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.TRIPLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.AROMATIC else 0.0

    ]
    is_conjugated = 1.0 if bond.GetIsConjugated() else 0.0
    '''Conjugation means, Electrons can move across multiple atoms (alternating double bonds).
       Important cuz increases chemical reactivity, affects drug absorption, affects toxicity strongly.
       Many toxic compounds are conjugated systems.'''

    is_in_ring = 1.0 if bond.IsInRing() else 0.0
    '''Checks whether bond belongs to a ring structure.
       Rings often: stabilize molecules, change metabolism, increase persistence in body.
       Many drugs and toxins contain rings.'''
    return np.array(bond_type_vec + [is_conjugated, is_in_ring], dtype=np.float32)

These feature definitions follow standard practice in molecular GNNs (e.g., Gilmer et al., 2017; Yang et al., 2019). Including atomic properties like symbol, degree, hybridization, and aromaticity captures essential chemical information, while bond type and conjugation (electrons are shared across multiple adjacent bonds) help the model understand connectivity and electronic effects.

Hybridization (Shape of electron orbitals)
| Type | Shape  | Example |
| ---- | ------ | ------- |
| sp   | linear | CO₂     |
| sp2  | flat   | benzene |
| sp3  | 3D     | methane |


## Convert SMILES to PyG Data Object

In [4]:
from torch_geometric.data import Data

def smiles_to_graph(smiles):
    """
    Convert a SMILES string to a PyTorch Geometric Data object.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Atom features (nodes)
    atom_feats = []
    for atom in mol.GetAtoms():
        atom_feats.append(atom_features(atom))
    x = torch.tensor(np.array(atom_feats), dtype=torch.float)

    # Bond features (edges)
    edge_indices = []
    edge_feats = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_indices += [[i, j], [j, i]]
        feats = bond_features(bond)
        edge_feats.append(feats)
        edge_feats.append(feats)  # both directions
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(np.array(edge_feats), dtype=torch.float)

    # Create Data object (target will be added later)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    return data

This function transforms a SMILES string into a graph representation that PyTorch Geometric can consume. It creates node features (x), edge indices (connectivity), and edge features (edge_attr). The graph is undirected, so each bond is stored twice (both directions).

## Process Unified Single‑Drug Dataset

 Load the unified CSV, convert each SMILES to a graph, and save a list of Data objects along with the corresponding pLD50 values.

In [5]:
import pandas as pd
import os
import pickle

base_path = '/content/drive/MyDrive/FYP/IRP/Data'
unified_csv = os.path.join(base_path, 'unified_single_drug_pld50.csv')
output_dir = os.path.join(base_path, 'processed_graphs')
os.makedirs(output_dir, exist_ok=True)

# Load unified dataset
df_unified = pd.read_csv(unified_csv)
print(f"Loaded {len(df_unified)} compounds.")

# Convert each SMILES to graph and store
graph_data = []
targets = []
failed = 0
for idx, row in df_unified.iterrows():
    smiles = row['SMILES']
    pld50 = row['pLD50']
    data = smiles_to_graph(smiles)
    if data is None:
        failed += 1
        continue
    data.y = torch.tensor([pld50], dtype=torch.float)
    graph_data.append(data)
    targets.append(pld50)

print(f"Successfully converted {len(graph_data)} molecules. Failed: {failed}")

# Save as a single file using torch.save (list of Data objects)
torch.save(graph_data, os.path.join(output_dir, 'unified_single_drug.pt'))

# Also save targets separately if needed
torch.save(torch.tensor(targets), os.path.join(output_dir, 'unified_targets.pt'))

Loaded 13639 compounds.
Successfully converted 13639 molecules. Failed: 0


Saving the processed graphs avoids recomputing them every time when train. The Data objects contain all graph information; the target y is attached to each object.

## Process MUDI Dataset (Interaction Learning)

In [6]:
mudi_train = os.path.join(base_path, 'MUDI/train_processed.csv')
mudi_test = os.path.join(base_path, 'MUDI/test_processed.csv')

def process_mudi_csv(csv_path, output_name):
    df = pd.read_csv(csv_path)
    graphs_a = []
    graphs_b = []
    labels = []
    failed = 0
    for idx, row in df.iterrows():
        smiles_a = row['smiles_a']
        smiles_b = row['smiles_b']
        label = row['label']
        data_a = smiles_to_graph(smiles_a)
        data_b = smiles_to_graph(smiles_b)
        if data_a is None or data_b is None:
            failed += 1
            continue
        graphs_a.append(data_a)
        graphs_b.append(data_b)
        labels.append(label)
    print(f"{output_name}: processed {len(labels)} pairs, failed {failed}")
    torch.save((graphs_a, graphs_b, torch.tensor(labels)), os.path.join(output_dir, output_name))
    return labels

labels_train = process_mudi_csv(mudi_train, 'mudi_train.pt')
labels_test = process_mudi_csv(mudi_test, 'mudi_test.pt')

mudi_train.pt: processed 221115 pairs, failed 0
mudi_test.pt: processed 89417 pairs, failed 0


For each drug pair, need two graph objects. Store them as two lists in a tuple. The interaction label is stored as a tensor.

In [7]:
df = pd.read_csv(mudi_train)
df['label'].value_counts()

,count
label,
0,186268
1,27320
2,7527


### Handle Class Imbalance

In [8]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights for training set
classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=labels_train)
class_weight_dict = {i: weights[i] for i in range(3)}
print("Class weights:", class_weight_dict)

# Save weights for later use
torch.save(weights, os.path.join(output_dir, 'mudi_class_weights.pt'))

Class weights: {0: np.float64(0.3956933021238216), 1: np.float64(2.697840409956076), 2: np.float64(9.792081838713964)}


So rare class → large weight

common class → small weight

If one interaction type is underrepresented, class weights in the loss function prevent the model from ignoring it. This follows best practices for imbalanced classification (He & Garcia, 2009).

## Process DDI LD₅₀ Smyth Dataset (Fine‑tuning)

In [9]:
smyth_csv = os.path.join(base_path, 'ddi_ld50_smyth1969/ddi_ld50_smyth1969_model_ready.csv')
df_smyth = pd.read_csv(smyth_csv)

graphs_a = []
graphs_b = []
targets = []
failed = 0
for idx, row in df_smyth.iterrows():
    smiles_a = row['SMILES_A']
    smiles_b = row['SMILES_B']
    mix_neglog = row['Mixture_neglog']
    data_a = smiles_to_graph(smiles_a)
    data_b = smiles_to_graph(smiles_b)
    if data_a is None or data_b is None:
        failed += 1
        continue
    graphs_a.append(data_a)
    graphs_b.append(data_b)
    targets.append(mix_neglog)

print(f"Smyth: processed {len(targets)} pairs, failed {failed}")
torch.save((graphs_a, graphs_b, torch.tensor(targets)), os.path.join(output_dir, 'smyth_ddi.pt'))

Smyth: processed 350 pairs, failed 0


The Smyth dataset is small and used for fine‑tuning. Storing it in the same format as MUDI allows reuse of data loaders.

## Target Standardization for Single‑Drug Pre‑training

In [13]:
from sklearn.preprocessing import StandardScaler

# Load targets from unified dataset
targets = torch.load(os.path.join(output_dir, 'unified_targets.pt')).numpy().reshape(-1,1)
scaler = StandardScaler()
scaler.fit(targets)
scaled_targets = scaler.transform(targets)
'''Each value is converted using: scaled=(value−mean)/std'''

# Save scaler for later inverse transformation
import joblib
joblib.dump(scaler, os.path.join(output_dir, 'pld50_scaler.pkl'))

# Replace y in graph objects with scaled value
graph_data = torch.load(os.path.join(output_dir, 'unified_single_drug.pt'), weights_only=False)
for i, data in enumerate(graph_data):
    data.y = torch.tensor(scaled_targets[i], dtype=torch.float)
torch.save(graph_data, os.path.join(output_dir, 'unified_single_drug_scaled.pt'))

Later, predictions must be converted back to real LD50 units.

Model output → inverse transform → real toxicity

Without saving this, predictions would stay in meaningless scaled units.

### Summary of Output Files

| File | Contents |
|------|--------|
| unified_single_drug.pt | List of PyG Data objects for single drugs (raw pLD₅₀) |
| unified_targets.pt | Tensor of raw pLD₅₀ values |
| unified_single_drug_scaled.pt	| Same graphs with scaled pLD₅₀ |
| pld50_scaler.pkl | Scaler used for inverse transformation of predictions |
| mudi_train.pt | Tuple containing (graphs_a, graphs_b, labels) for training |
| mudi_test.pt | Tuple containing (graphs_a, graphs_b, labels) for testing |
| mudi_class_weights.pt | Class weight tensor for handling class imbalance |
| smyth_ddi.pt | Tuple containing (graphs_a, graphs_b, targets) for regression DDI dataset |
